# TMDB 5000 Movie Dataset으로 장르기반 추천시스템 만들기
* 장르에 있는 텍스트를 벡터화 후 텍스트간 유사도를 구해 가까운 순서대로 정렬
* 유사도 척도: 코사인유사도

In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib

In [80]:
data = pd.read_csv("https://raw.githubusercontent.com/haram4th/ablearn/main/tmdb_5000_movies.csv")
data.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [81]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [82]:
data['genres'][0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [83]:
import json

In [84]:
json.loads(data['genres'][0])[0]['name']

'Action'

In [85]:
data.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [86]:
dict_cols = ['genres', 'keywords', 'production_companies', 'production_countries', 'spoken_languages']
dict_cols

['genres',
 'keywords',
 'production_companies',
 'production_countries',
 'spoken_languages']

In [87]:
def dict_unpack(data):
    for col in dict_cols:
        temp = []
        for i in json.loads(data[col]):
            temp.append(i.get('name', 'None'))
        # print(temp)
        temp = " ".join(temp)
        print(temp)
        data[col] = temp
    return data

In [88]:
data.apply(dict_unpack, axis=1)

Action Adventure Fantasy Science Fiction
culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d
Ingenious Film Partners Twentieth Century Fox Film Corporation Dune Entertainment Lightstorm Entertainment
United States of America United Kingdom
English Español
Adventure Fantasy Action
ocean drug abuse exotic island east india trading company love of one's life traitor shipwreck strong woman ship alliance calypso afterlife fighter pirate swashbuckler aftercreditsstinger
Walt Disney Pictures Jerry Bruckheimer Films Second Mate Productions
United States of America
English
Action Adventure Crime
spy based on novel secret agent sequel mi6 british secret service united kingdom
Columbia Pictures Danjaq B24
United Kingdom United States of America
Français English Español Italiano Deutsch
Action Crime Drama Thriller
dc comics crime fighter terrorist secret id

English
Animation Comedy Family
weather food science
Columbia Pictures Sony Pictures Animation
United States of America
English
Animation Comedy Family Adventure
ice age bridge insanity jungle dinosaur birth duringcreditsstinger 3d
Blue Sky Studios Twentieth Century Fox Animation
United States of America
English
Adventure Comedy Drama Fantasy
himalaya photographer magazine iceland daydream photograph shark fired from the job skateboard dreamer online dating daydreaming
New Line Cinema Ingenious Media Twentieth Century Fox Film Corporation Samuel Goldwyn Films Big Screen Productions Red Hour Films TSG Entertainment Down Productions
United States of America
English
Action Adventure Comedy Crime Thriller
martial arts female friendship millionaire agent
Columbia Pictures
United States of America
English Deutsch suomi 日本語 广州话 / 廣州話
Drama Thriller Crime
undercover boston police friends mafia undercover cop mobster mole state police police training realtor
Vertigo Entertainment Media Asia Fil

dragon evolution fire chief animated map theatre audience dragonslayer tunnel construction fire repellent drilling iodine
The Zanuck Company Spyglass Entertainment World 2000 Entertainment Touchstone Pictures Tripod Entertainment
Ireland United Kingdom United States of America
English
Crime Drama Action Thriller
los angeles gangster
Village Roadshow Pictures Lin Pictures Warner Bros. Langley Park Production
United States of America
English
Comedy Adventure
temple slavery stone age circumcision hebrews cavemen prehistoric adventure duringcreditsstinger prehistoric times prehistoric man
Columbia Pictures Ocean Pictures Apatow Productions
United States of America
English
Drama History
stadium south africa apartheid nelson mandela sport nation rugby president racism poverty celebration duringcreditsstinger
Spyglass Entertainment Malpaso Productions Revelations Entertainment Mace Neufeld Productions Warner Bros. Liberty Pictures (II)
United States of America
English
Action
corruption assass

Drama History Romance
ancient rome historical figure cleopatra julius caesar
Twentieth Century Fox Film Corporation MCL Films S.A. Walwa Films S.A.
United Kingdom United States of America Switzerland
English Português
Comedy
high school teacher prize money physics teacher fighting movie budget cutting
Columbia Pictures Happy Madison Productions Broken Road Productions
United States of America
English
Drama Mystery Thriller Crime
based on novel witness village court love murder lawyer defense trial justice husband u.s. marine arrested classified
Twentieth Century Fox Film Corporation Regency Enterprises Monarch Pictures Epsilon Motion Pictures New Regency Pictures Manifest Film Company
United States of America
English Español
Comedy Drama Romance
sex professor wedding woman director columbia university
TriStar Pictures Phoenix Pictures
United States of America
English
Drama Horror Mystery
based on novel small town dream motel hallucination bridge alien life-form warning tumor west virgi

United States of America
English Italiano Português
Drama
diving u.s. navy
Fox 2000 Pictures
United States of America
English
Action Crime Drama Thriller
heist
Rainforest Films
United States of America
English
Comedy
adoption marriage divorce birth mother
Millenium Films Two Ton Films
United States of America
English Español Latin
Crime Comedy Action
undercover fbi sequel comedy disguise fbi agent impersonation duringcreditsstinger
Twentieth Century Fox Film Corporation Regency Enterprises New Regency Pictures The Collective Studios Runteldat Entertainment Friendly Films (II) Friendly Films Productions
United States of America
English
Thriller Science Fiction Mystery
bomb identity fantasy bomber suspicion time travel investigation surrealism soldier helicopter pilot
The Mark Gordon Company Vendome Pictures
Canada United States of America
English
Action Adventure Drama Thriller
rugby stranded survival plane wreck airplane crash freezing disaster movie
Paramount Pictures Touchstone Pictu

...

Drama
civil war dictator journalist guerrilla loss of lover revolution war correspondent civil rights movement  picture journalist el salvador dictatorship
Hemdale Film
United Kingdom United States of America
English Español
Comedy
wife husband relationship stress children mother daughter relationship parenthood parenting parent child relationship duringcreditsstinger
TriStar Pictures Provident Films Pure Flix Entertainment Affirm Films FourBoys Entertainment
United States of America
English
Fantasy Drama Mystery
parents kids relationship airplane time travel school presentation school performance suburbia vision morality teenager
Pandora Cinema Flower Films Adam Fields Productions
United States of America
English
Action Comedy Foreign




Drama Foreign History
law and ethics
First Floor Features Almerica Films
Netherlands Belgium
Nederlands English Français Deutsch
Comedy Crime Drama
independent film speed freak junky cop boy toy porn magazine dope selling crank handcuffed to a bed cc

Drama Romance
paris journalist dialogue talking soulmates walking bookshop love of one's life author
Castle Rock Entertainment Detour Film Production Warner Independent Pictures (WIP)
United States of America
English Français
Drama Thriller
homeless person mexico city daughter secret love dogfight money dog nonlinear timeline multiple storylines new mexican cinema
Altavista Films Zeta Film
Mexico
Español
Crime Drama
cheating dysfunctional family teen angst underage drinking domestic violence makeover drug overdose teacher student relationship street life movie theater razor blade tattoo shop peer pressure shoe store overachiever reference to jack black flunking out of school promiscuous mother glue sniffing woman director
Fox Searchlight Pictures Sound for Film Working Title Films Antidote Films (I)
United States of America
English Português Español
Drama Romance
anti semitism soldier
Twentieth Century Fox Film Corporation
United States of America
English
Drama
father court bail drug t



United States of America
English
Horror
racist religion
Splendid Film
United States of America
English
Horror Thriller



English
Romance Drama

Baleuko S.L. Departamento de Cultura del Gobierno Vasco Bitart New Media
Spain
Español
Comedy Drama Romance


United States of America
English
Comedy Romance

Triumphant Pictures
United States of America
English
Western

Rapid Heart Pictures
United States of America

Thriller Horror
friends remote island woman director
Submarine Entertainment Distributors LD Entertainment
United States of America
English
Horror Thriller Mystery

Corona Pictures
United Kingdom
English
Drama Thriller
independent film


Deutsch
Science Fiction Drama Music
small town rock star musical idol teenager 1960s chemical leak mutations
JoBro Productions & Film Finance Scythia Films
Canada
English
Horror
phobia doctor fear
Dry County Films Anchor Bay Entertainment Movie Machine
United States of America
English
Comedy Action Science Fiction Thriller
mutant post-apocalypti




English
Horror
high school murder slasher teenager


English
Documentary Music
1970s music
FM Productions Last Waltz Inc.
United States of America
English
Thriller Horror
revenge murder
Little Big Film Company Abundant Productions Faith vs. Fate Productions
United States of America
English
Drama Romance
independent film


English
Action Western




Drama Comedy
musician romance independent film mumblecore


English
Horror




Drama Action Comedy
murder dark comedy crime family

United Kingdom
English
Comedy
salesclerk loser aftercreditsstinger
Miramax Films View Askew Productions
United States of America
English
Drama Romance
dream prostitution
Strand Releasing
United States of America
English Français
Drama Comedy
mumblecore

United States of America
English
Comedy Drama
office love independent film secretary misogynist
Alliance Atlantis Communications Fair and Square Productions
Canada United States of America
English
Drama


United States of America
Español English
Action Drama C

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,Ingenious Film Partners Twentieth Century Fox ...,United States of America United Kingdom,2009-12-10,2787965087,162.0,English Español,Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,Walt Disney Pictures Jerry Bruckheimer Films S...,United States of America,2007-05-19,961000000,169.0,English,Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6 bri...,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,Columbia Pictures Danjaq B24,United Kingdom United States of America,2015-10-26,880674609,148.0,Français English Español Italiano Deutsch,Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,Legendary Pictures Warner Bros. DC Entertainme...,United States of America,2012-07-16,1084939099,165.0,English,Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,Walt Disney Pictures,United States of America,2012-03-07,284139100,132.0,English,Released,"Lost in our world, found in another.",John Carter,6.1,2124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4798,220000,Action Crime Thriller,NaN,9367,united states–mexico barrier legs arms paper k...,es,El Mariachi,El Mariachi just wants to play his guitar and ...,14.269792,Columbia Pictures,Mexico United States of America,1992-09-04,2040920,81.0,Español,Released,"He didn't come looking for trouble, but troubl...",El Mariachi,6.6,238
4799,9000,Comedy Romance,NaN,72766,,en,Newlyweds,A newlywed couple's honeymoon is upended by th...,0.642552,,,2011-12-26,0,85.0,,Released,A newlywed couple's honeymoon is upended by th...,Newlyweds,5.9,5
4800,0,Comedy Drama Romance TV Movie,http://www.hallmarkchannel.com/signedsealeddel...,231617,date love at first sight narration investigati...,en,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...",1.444476,Front Street Pictures Muse Entertainment Enter...,United States of America,2013-10-13,0,120.0,English,Released,NaN,"Signed, Sealed, Delivered",7.0,6
4801,0,,http://shanghaicalling.com/,126186,,en,Shanghai Calling,When ambitious New York attorney Sam is sent t...,0.857008,,United States of America China,2012-05-03,0,98.0,English,Released,A New Yorker in Shanghai,Shanghai Calling,5.7,7


In [89]:
data.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [90]:
data = data[['original_title', 'genres', 'popularity', 'runtime', 'vote_average', 'vote_count']]

In [91]:
data = data.dropna()
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4801 entries, 0 to 4802
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   original_title  4801 non-null   object 
 1   genres          4801 non-null   object 
 2   popularity      4801 non-null   float64
 3   runtime         4801 non-null   float64
 4   vote_average    4801 non-null   float64
 5   vote_count      4801 non-null   int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 262.6+ KB


# 장르기반 추천시스템 만들기
* 장르 컬럼의 텍스트를 숫자로 벡터화
* 코사인 유사도를 구해 비슷한 장르의 행 검색
* 유사도가 높은 행을 기준으로 내림차순 정렬

In [92]:
from sklearn.feature_extraction.text import CountVectorizer

In [93]:
cv = CountVectorizer(ngram_range=(1,2))
genres_mat = cv.fit_transform(data['genres'])
print(len(cv.get_feature_names_out()))
genres_mat_df = pd.DataFrame(genres_mat)
genres_mat_df

126


,0
0,"(np.int32(0), np.int32(66))\t4\n (np.int32(..."
1,"(np.int32(0), np.int32(66))\t3\n (np.int32(..."
2,"(np.int32(0), np.int32(66))\t3\n (np.int32(..."
3,"(np.int32(0), np.int32(66))\t4\n (np.int32(..."
4,"(np.int32(0), np.int32(66))\t3\n (np.int32(..."
...,...
4796,"(np.int32(0), np.int32(66))\t3\n (np.int32(..."
4797,"(np.int32(0), np.int32(66))\t2\n (np.int32(..."
4798,"(np.int32(0), np.int32(66))\t4\n (np.int32(..."
4799,


In [94]:
genres_mat_df.loc[0,:]

0      (np.int32(0), np.int32(66))\t4\n  (np.int32(...
Name: 0, dtype: object

* 코사인 유사도를 이용해 genres_mat의 유사도 산출 후 genres_sim 생성
* 비슷한 장르의 영화를 찾기 위해서 계산
* 코사인 유사도는 주로 문자, 문서의 유사도를 계산하는데 사용 됨

In [97]:
from sklearn.metrics.pairwise import cosine_similarity

In [98]:
genres_sim = cosine_similarity(genres_mat, genres_mat)
print(genres_sim.shape)

(4801, 4801)


In [99]:
genres_sim_df = pd.DataFrame(genres_sim)
genres_sim_df

,0,1,2,3,4,5,6,7,8,9,...,4791,4792,4793,4794,4795,4796,4797,4798,4799,4800
0,1.000000,0.917936,0.805993,0.678680,0.936333,0.917936,0.486190,0.936333,0.805993,0.917936,...,0.400501,0.537328,0.400501,0.675031,0.486190,0.671660,0.486190,0.561404,0.0,0.400501
1,0.917936,1.000000,0.828571,0.660971,0.805867,0.971429,0.465340,0.805867,0.857143,0.971429,...,0.383326,0.514286,0.383326,0.500193,0.465340,0.657143,0.465340,0.537328,0.0,0.383326
2,0.805993,0.828571,1.000000,0.797724,0.833655,0.828571,0.465340,0.833655,0.685714,0.857143,...,0.383326,0.514286,0.383326,0.500193,0.465340,0.828571,0.465340,0.537328,0.0,0.383326
3,0.678680,0.660971,0.797724,1.000000,0.665027,0.683763,0.494951,0.665027,0.547011,0.683763,...,0.662541,0.660971,0.662541,0.775864,0.649623,0.934477,0.494951,0.678680,0.0,0.407718
4,0.936333,0.805867,0.833655,0.665027,1.000000,0.805867,0.452589,1.000000,0.666924,0.833655,...,0.372822,0.500193,0.372822,0.675676,0.452589,0.666924,0.452589,0.522604,0.0,0.372822
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4796,0.671660,0.657143,0.828571,0.934477,0.666924,0.685714,0.465340,0.666924,0.514286,0.685714,...,0.383326,0.657143,0.383326,0.639136,0.659232,1.000000,0.465340,0.537328,0.0,0.383326
4797,0.486190,0.465340,0.465340,0.494951,0.452589,0.465340,0.421053,0.452589,0.465340,0.465340,...,0.346844,0.659232,0.346844,0.452589,0.421053,0.465340,1.000000,0.820445,0.0,0.346844
4798,0.561404,0.537328,0.537328,0.678680,0.522604,0.537328,0.486190,0.522604,0.537328,0.537328,...,0.650814,0.649272,0.650814,0.653255,0.486190,0.537328,0.820445,1.000000,0.0,0.400501
4799,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000


In [101]:
sorted_genres_sim = genres_sim.argsort()[:, ::-1]

In [102]:
sorted_genres_sim[:1]

array([[  14,  813,  870, ..., 4609, 4655, 4714]])

In [103]:
data.loc[0,:]

original_title                                               Avatar
genres            [{"id": 28, "name": "Action"}, {"id": 12, "nam...
popularity                                               150.437577
runtime                                                       162.0
vote_average                                                    7.2
vote_count                                                    11800
Name: 0, dtype: object

In [104]:
data.loc[14,:]

original_title                                         Man of Steel
genres            [{"id": 28, "name": "Action"}, {"id": 12, "nam...
popularity                                                99.398009
runtime                                                       143.0
vote_average                                                    6.5
vote_count                                                     6359
Name: 14, dtype: object

In [105]:
data.loc[813,:]

original_title                                             Superman
genres            [{"id": 28, "name": "Action"}, {"id": 12, "nam...
popularity                                                48.507081
runtime                                                       143.0
vote_average                                                    6.9
vote_count                                                     1022
Name: 813, dtype: object

In [106]:
data.loc[870,:]

original_title                                          Superman II
genres            [{"id": 28, "name": "Action"}, {"id": 12, "nam...
popularity                                                30.515175
runtime                                                       127.0
vote_average                                                    6.5
vote_count                                                      629
Name: 870, dtype: object

# 영화 이름과 추천 받을 개수를 입력받아 추천영화 출력하기

In [109]:
movie_index = data[data['original_title'] == 'Avatar'].index.values[0]

In [111]:
recommaned_idx = sorted_genres_sim[movie_index, 1:11]

In [112]:
data.loc[recommaned_idx, :]

,original_title,genres,popularity,runtime,vote_average,vote_count
813,Superman,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",48.507081,143.0,6.9,1022
870,Superman II,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",30.515175,127.0,6.5,629
3493,Morvern Callar,"[{""id"": 18, ""name"": ""Drama""}]",2.507912,97.0,7.2,34
46,X-Men: Days of Future Past,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",118.078691,131.0,7.5,6032
0,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",150.437577,162.0,7.2,11800
10,Superman Returns,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",57.925623,154.0,5.4,1400
232,The Wolverine,"[{""id"": 28, ""name"": ""Action""}, {""id"": 878, ""na...",15.953444,126.0,6.3,4053
61,Jupiter Ascending,"[{""id"": 878, ""name"": ""Science Fiction""}, {""id""...",85.369080,124.0,5.2,2768
3207,Awake,"[{""id"": 53, ""name"": ""Thriller""}, {""id"": 80, ""n...",18.172736,84.0,6.3,395
1296,Superman III,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 28, ""nam...",22.164202,125.0,5.3,490


In [115]:
def find_sim_movie(data, sorted_genres_sim, title_name, top_n=10):
    title_movie = data[data['original_title'] == title_name]
    title_index = title_movie.index.values[0]
    similar_idxs = sorted_genres_sim[title_index, 1:top_n+1]
    return data.iloc[similar_idxs]

In [116]:
find_sim_movie(data, sorted_genres_sim, "Superman III", 5)

,original_title,genres,popularity,runtime,vote_average,vote_count
1932,Sheena,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",4.020194,117.0,5.0,22
618,Mystery Men,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",12.948627,121.0,5.7,250
238,Teenage Mutant Ninja Turtles,"[{""id"": 878, ""name"": ""Science Fiction""}, {""id""...",143.350376,101.0,5.8,2636
1191,Small Soldiers,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 12, ""nam...",23.088571,110.0,6.2,511
859,Thunderbirds,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",9.278750,95.0,4.2,91
